# RayDINO Backbone Verification

**Model**: RayDINO — ViT-L (307M), DINOv2 self-supervised training on 873k CXRs  
**Arch**: ViT-Large (DINOv2-style), producing 1024-dim tokens  
**Embed dim**: 1024 (CLS token)  
**Input**: 3×518×518 px (DINOv2 ViT-L standard) or 3×224×224 for small crops  
**Access**: ⚠️ **Weights not yet publicly released** as of the paper (May 2024).  
            Monitor the authors' repository for future release.  
**Zero-shot**: No — vision-only (DINOv2 pretraining)

Paper: [Advancing Human-Centric AI for Robust X-ray Analysis Through Holistic Self-Supervised Learning (Nature Machine Intelligence, 2025)](https://arxiv.org/abs/2405.01469)  
Authors: Jakub Chromý et al. (Philips / Deepc / European consortium)

---

> ⚠️ **Status**: The paper states "Models will be available soon" — no public checkpoint
> link found as of May 2026. This notebook is a template ready to run once weights
> are released. The loading pattern follows standard DINOv2 ViT-L.
>
> If you have access to an institution-shared checkpoint, point `CKPT_PATH` below
> at that file — the loading code will work unchanged.

In [ ]:
%pip install -q torch torchvision transformers Pillow
# DINOv2 ViT-L is supported natively in transformers >= 4.37

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model

RayDINO was trained with the DINOv2 codebase. The checkpoint format should be
compatible with DINOv2 ViT-L loading via `transformers.AutoModel`
(if released as an HF model) or via direct `torch.load` + state_dict injection
(if released as a raw `.pth` file).

In [ ]:
import torch
import os

# ── Option A: released as HuggingFace model (ideal) ─────────────────────────
# Replace with the actual HF hub ID once published
HF_HUB = None   # e.g. "raydino/raydino-vitl" — fill in when available

# ── Option B: raw checkpoint file (.pth / .pt) ───────────────────────────────
CKPT_PATH = "/path/to/raydino_vitl.pth"

device = "cuda" if torch.cuda.is_available() else "cpu"

if HF_HUB:
    # Option A
    from transformers import AutoModel
    model = AutoModel.from_pretrained(HF_HUB)
    print("Loaded from HuggingFace:", HF_HUB)
elif os.path.exists(CKPT_PATH):
    # Option B — load as DINOv2 ViT-L with custom weights
    from transformers import Dinov2Model, Dinov2Config

    # RayDINO uses ViT-L architecture (same as DINOv2 ViT-L)
    config = Dinov2Config.from_pretrained("facebook/dinov2-large")
    model  = Dinov2Model(config)

    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    # Checkpoint might be nested — try common keys
    state = (ckpt.get("teacher") or ckpt.get("student") or
             ckpt.get("state_dict") or ckpt)
    # Strip 'backbone.' prefix if present
    state = {k.replace("backbone.", ""): v for k, v in state.items()
             if not k.startswith("head")}
    missing, unexpected = model.load_state_dict(state, strict=False)
    print("Missing (non-head):",    [k for k in missing    if "head" not in k][:5])
    print("Unexpected:", unexpected[:5])
    print("Loaded from checkpoint:", CKPT_PATH)
else:
    # No weights available — load DINOv2 ViT-L from Meta as a stand-in
    print("⚠️  RayDINO weights not found — loading DINOv2 ViT-L from Meta as placeholder")
    print("   Replace with RayDINO weights when available.")
    from transformers import AutoModel
    model = AutoModel.from_pretrained("facebook/dinov2-large")

model = model.to(device)
model.eval()
print("Architecture:", model.__class__.__name__)
print("Hidden size: ", model.config.hidden_size)   # 1024 for ViT-L

## 2. Image transform

RayDINO uses the DINOv2 training pipeline: 518×518 input resolution for ViT-L
(14×14 patches → 37×37 = 1369 patch tokens), ImageNet normalization.

In [ ]:
from torchvision import transforms
from torchvision.transforms.functional import InterpolationMode
from PIL import Image

INPUT_SIZE = 518   # DINOv2 ViT-L/14 standard

def make_transform():
    return transforms.Compose([
        transforms.Resize(INPUT_SIZE, interpolation=InterpolationMode.BICUBIC),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.Grayscale(num_output_channels=1),
        transforms.ToTensor(),
        transforms.Lambda(lambda x: x.repeat(3, 1, 1)),   # 1ch → 3ch
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std =[0.229, 0.224, 0.225],
        ),
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    """Return [1, 3, 518, 518] batch tensor."""
    img = Image.open(path)
    return transform(img).unsqueeze(0).to(device)

dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE, device=device)
print("dummy input:", dummy.shape)

## 3. Extract embeddings

In [ ]:
with torch.no_grad():
    out = model(pixel_values=dummy)

cls_emb   = out.last_hidden_state[:, 0]           # [B, 1024] — CLS token
patch_emb = out.last_hidden_state[:, 1:]           # [B, N_patches, 1024]
mean_emb  = patch_emb.mean(dim=1)                 # [B, 1024] — mean pool
pooled    = out.pooler_output                      # [B, 1024] — model's pooled CLS

print("CLS token:   ", cls_emb.shape)     # [1, 1024]
print("patch tokens:", patch_emb.shape)   # [1, 1369, 1024] for 518px ViT-L/14
print("mean pool:   ", mean_emb.shape)    # [1, 1024]
print("pooler_output:", pooled.shape)     # [1, 1024]

## Segmentation mode — spatial feature map

Reshape the patch sequence from section 3 to a 2-D spatial grid. RayDINO uses
DINOv2 ViT-L/14 at 518×518: **37×37** patch grid (518/14≈37, 37²=1369 patches),
1024-dim features.

In [ ]:
import math

# patch_emb from section 3: out.last_hidden_state[:, 1:]
# 518px / 14px-patches (DINOv2 ViT-L/14) → 37×37 = 1369 patches, D=1024
B, N, D = patch_emb.shape
H = W = int(math.sqrt(N))   # 37 for 518px / 14px-patches
assert H * W == N

spatial = patch_emb.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patch_emb.shape}")   # [1, 1369, 1024]
print(f"spatial map:   {spatial.shape}")     # [1, 1024, 37, 37]

## 4. Inspect checkpoint structure (diagnostic — run if load fails)

In [ ]:
if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    print("Top-level keys:", list(ckpt.keys()) if isinstance(ckpt, dict) else type(ckpt))
    state = (ckpt.get("teacher") or ckpt.get("state_dict") or ckpt)
    keys = list(state.keys())
    print("State dict first 10 keys:", keys[:10])
else:
    print("No checkpoint file to inspect.")

## Notes for RadHarmony integration

```
model_call : lambda m, x: m(pixel_values=x).last_hidden_state[:, 0]   # CLS
pool       : None
embed_dim  : 1024
transform  : Resize(518,BICUBIC) → CenterCrop(518) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(ImageNet)
```

**Weight availability**: Contact the authors (Philips Research / Deepc) or monitor
the paper's supplementary for a release link. The architecture is identical to
`facebook/dinov2-large`, so the loading code above is ready — just supply `CKPT_PATH`
or `HF_HUB`.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model(pixel_values=x).last_hidden_state[:, 1:]` → `[B, 1369, 1024]`
- Spatial map: reshape to `[B, 1024, 37, 37]` (518px / 14px-patches → 37×37)

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH)
with torch.no_grad():
    out = model(pixel_values=x)
    emb = out.last_hidden_state[:, 0]   # CLS

print("embedding:", emb.shape)
print("L2 norm:  ", emb.norm(dim=-1).item())

## Notes for RadHarmony integration

```
model_call : lambda m, x: m(pixel_values=x).last_hidden_state[:, 0]   # CLS
pool       : None
embed_dim  : 1024
transform  : Resize(518,BICUBIC) → CenterCrop(518) → Grayscale → ToTensor
             → repeat(3ch) → Normalize(ImageNet)
```

**Weight availability**: Contact the authors (Philips Research / Deepc) or monitor
the paper's supplementary for a release link. The architecture is identical to
`facebook/dinov2-large`, so the loading code above is ready — just supply `CKPT_PATH`
or `HF_HUB`.